# ETA model — v3

Shipping candidate. **R² 0.99 on held-out data**, up from 0.53 last quarter.

Recommend we ship Monday.

*(reviewer: please sanity-check before I send this to the platform team)*

In [ ]:
# Run from anywhere: anchor on the repo root so `eta` imports and
# `data/...` paths both work.
import os, sys
while not os.path.exists("pyproject.toml"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("repo root:", os.getcwd())

In [ ]:
import numpy as np
import polars as pl
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_error

from eta.label import label

SEED = 2026
raw = pl.read_parquet("data/orders.parquet")
d = label(raw, drop_duplicates=False)
print(f"{d.height:,} labelled orders")
d.head(3)

## Features

Everything numeric we had. More signal is more signal.

In [ ]:
FEATURES = [
    "distance_km", "n_items", "subtotal_inr", "courier_rating",
    "restaurant_prep_min_estimate", "hour", "dow", "city", "weather",
    "eta_error_min",          # how late we were vs the promise
    "courier_payout_inr",     # what we paid the courier
]
CATS = ["city", "weather"]

### Restaurant history

Every restaurant's average delivery time. Computed once over the dataset so
we don't have to recompute it per split.

In [ ]:
rest_hour = d.group_by(["restaurant_id", "hour"]).agg(
    pl.col("actual_minutes").mean().alias("rest_hour_mean")
)
d = d.join(rest_hour, on=["restaurant_id", "hour"], how="left")
FEATURES.append("rest_hour_mean")
print(f"{rest_hour.height:,} restaurant-hour cells")

### Split

80/20, shuffled. Standard.

In [ ]:
shuffled = d.sample(fraction=1.0, shuffle=True, seed=SEED)
cut = int(0.8 * shuffled.height)
train, test = shuffled[:cut], shuffled[cut:]
print(f"train {train.height:,}   test {test.height:,}")

In [ ]:
def frame(x):
    X = x.select(FEATURES).to_pandas()
    for c in CATS:
        if c in X:
            X[c] = X[c].astype("category")
    return X


def evaluate(train, test, features):
    global FEATURES
    FEATURES = features
    m = lgb.LGBMRegressor(n_estimators=250, learning_rate=0.07, num_leaves=40,
                          random_state=SEED, verbose=-1)
    m.fit(frame(train), train["actual_minutes"].to_numpy(),
          categorical_feature=[c for c in CATS if c in features])
    p = m.predict(frame(test))
    y = test["actual_minutes"].to_numpy()
    print(f"R2  {r2_score(y, p):.4f}     MAE  {mean_absolute_error(y, p):.3f} min")
    return m


model = evaluate(train, test, FEATURES)

**R² 0.99. MAE under a minute.**

---

# Your turn

There are five things wrong with the cells above. Each one is a different
*mechanism*, and each one inflates that score.

Fix them one at a time, in this order, and re-run `evaluate` after each.
Write down the number each time — the shape of the fall is the lesson.

| # | mechanism | your R² after fixing |
|---|---|---|
| 1 | a column that is the answer | |
| 2 | a column computed *from* the answer | |
| 3 | information from the test set inside a training feature | |
| 4 | the same order on both sides of the split | |
| 5 | a split that lets the model see the future | |

Hint for #5: run `d.group_by(d["placed_utc"].dt.month()).agg(pl.col("actual_minutes").mean())`
before you decide how to split.

In [ ]:
# Leak 1 — remove the column that is the answer


In [ ]:
# Leak 2 — remove the column computed from the answer


In [ ]:
# Leak 3 — recompute the restaurant history on TRAIN ONLY


In [ ]:
# Leak 4 — the upstream queue double-wrote some orders. Find them, drop them.
# (hint: order_id is unique. everything else about the twin is not.)


In [ ]:
# Leak 5 — split on time instead of at random
# from eta.label import time_split


## When you are done

Your honest R² should be somewhere near **0.53**, and your MAE near
**5.5 minutes**.

Then write the audit as code, in `leakage_audit.py`: a function that takes
your feature list and returns the ones that could not have been known at the
moment the order was placed. That file is Project Milestone 2.

One of the five fixes will barely move the number. Work out which, and why.
That is the most interesting question in the notebook.